In [1]:
# 1. Install the specific 'prebuilt' support package
%pip install -U langgraph-prebuilt

# 2. Re-sync the main packages to ensure they match
%pip install -U langchain langgraph langchain-groq

%pip install python-dotenv

%pip install tavily

%pip install langchain-community sqlalchemy

  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.7
    Uninstalling langgraph-prebuilt-1.0.7:
      Successfully uninstalled langgraph-prebuilt-1.0.7

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.4/501.4 kB 16.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
    Uninstalling langchain-core-1.2.9:
      Successfully uninstalled langchain-core-1.2.9
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.8
    Uninstalling langgraph-1.0.8:
      Successfully uninstalled langgraph-1.0.8
  Attempting uninstall: langchain

In [11]:
! pip3 install pypdf
%pip install langchain-openai


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 9.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.6/318.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.1/289.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.tools import ToolRuntime
from langchain.agents import create_agent, AgentState
from langgraph.types import Command
from langchain_core.prompts import ChatPromptTemplate
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.utilities import SQLDatabase

from tavily import TavilyClient

from typing import Dict, Any
from pprint import pprint
from dataclasses import dataclass
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

### Run time context

In [67]:
@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [68]:
@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [7]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile" ,
    groq_api_key=os.getenv("OPENAI_API_KEY") ,
    temperature=0 ,
)

In [70]:
agent = create_agent(
    model=llm,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext,
)

In [71]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)
print(f"AI response: {response["messages"][-1].content}")

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='39b6b9bf-f297-4687-b1a0-9d5a432da48d'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dgj7kdra3', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 267, 'total_tokens': 277, 'completion_time': 0.028571922, 'completion_tokens_details': None, 'prompt_time': 0.013565287, 'prompt_tokens_details': None, 'queue_time': 0.048617033, 'total_time': 0.042137209}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c46ee-85be-73c3-a748-014891e9cff2-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'dgj7kdra3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


In [72]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='f2d1bd04-3ea7-4bdc-9823-d8464ea121d2'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4gy5vtxbe', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 267, 'total_tokens': 277, 'completion_time': 0.027859848, 'completion_tokens_details': None, 'prompt_time': 0.026370723, 'prompt_tokens_details': None, 'queue_time': 0.049614797, 'total_time': 0.054230571}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c46ee-870c-7150-a4ef-eaffdade4fe4-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': '4gy5vtxbe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


### State

In [73]:
class CustomState(AgentState):
    favourite_colour: str

In [74]:
@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

In [75]:
state_agent = create_agent(
    model=llm,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

In [76]:
response = state_agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

Your favourite colour has been successfully updated to green and it matches the colour currently stored in the state.


In [77]:
response = state_agent.invoke(
    { "messages": [HumanMessage(content="What is my favourite color?")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

Your favourite colour is green.


### Subagent

In [78]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [79]:
subagent_1 = create_agent(
    model=llm,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=llm,
    tools=[square]
)

In [80]:
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=llm,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

In [81]:
response = main_agent.invoke({"messages": [HumanMessage(content="What is the square root of 456?")]})

print(response["messages"][-1].content)

The square root of 456 is 21.354156504062622.


### Access db

In [3]:
@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

In [4]:
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

In [5]:
tools = [web_search, query_playlist_db]

In [8]:
# Travel agent
travel_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

# Venue agent
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    """
)

# Playlist agent
playlist_agent = create_agent(
    model=llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.
    You may need to make multiple queries to iteratively find the best options.
    """
)

In [10]:
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    try:
        origin = runtime.state["origin"]
        destination = runtime.state["destination"]
    except KeyError:
        return "Error: Origin and destination are not set in the state. Please call update_state first."

    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    try:
        destination = runtime.state["destination"]
        capacity = runtime.state["guest_count"]
    except KeyError:
        return "Error: Destination and guest_count are not set in the state. Please call update_state first."

    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    try:
        genre = runtime.state["genre"]
    except KeyError:
        return "Error: Genre is not set in the state. Please call update_state first."

    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre"""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [11]:
coordinator = create_agent(
    model=llm,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues and playlists.
    First find all the information you need to update the state. Once that is done you can delegate the tasks.
    Once you have received their answers, coordinate the perfect wedding for me.
    
    IMPORTANT: You must use the tools provided to you. Do not hallucinate tool output or use XML tags. Use standard tool calling.
    """
)

In [15]:
config = {"recursion_limit": 100}

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config=config,
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kf5zfk82eaa98vfv8sepa7rd` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99800, Requested 531. Please try again in 4m45.984s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre", additional_kwargs={}, response_metadata={}, id='327f06d1-af30-4b13-9e06-5271a3aa05b1'),
              AIMessage(content='<function(update_state {"origin": "London", "destination": "Paris", "guest_count": "100", "genre": "jazz"})</function>\n<function(search_flights)</function>\n<function(search_venues)</function>\n<function(suggest_playlist)</function>', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 502, 'total_tokens': 560, 'completion_time': 0.138319347, 'completion_tokens_details': None, 'prompt_time': 0.027518535, 'prompt_tokens_details': None, 'queue_time': 0.049953636, 'total_time': 0.165837882}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c4681-9285-7862-a1a2-ba3d4f9d7

In [ ]:
print(response["messages"][-1].content)

<function(update_state {"origin": "London", "destination": "Paris", "guest_count": "100", "genre": "jazz"})</function>
<function(search_flights)</function>
<function(search_venues)</function>
<function(suggest_playlist)</function>


# RAG

In [8]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

incorrect startxref pointer(1)
parsing for Object Streams


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

3


In [12]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [13]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [14]:
ids = vector_store.add_documents(documents=all_splits)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: gsk_rRxY********************************************F7DZ. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}